In [1]:
# ============================================
# EEG Modelling — Cell 1
# Setup + loading + subject-wise split + dataloaders
# ============================================

import os
import random
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------------
# Reproducibility
# -------------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# -------------------------------
# Device
# -------------------------------
# -------------------------------
# Device
# -------------------------------
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("Detected GPU:", gpu_name)
    device = torch.device("cuda")
else:
    gpu_name = "CPU"
    print("Detected GPU: CPU")
    device = torch.device("cpu")

PIN_MEMORY = (device.type == "cuda")
print("Using device:", device)
# -------------------------------
# Paths
# -------------------------------
DATA_DIR = Path("/kaggle/input/datasets/katakuricharlotte/eeg-preprocessing-and-epoching-results/eeg_preproc_outputs")
OUT_DIR = Path("/kaggle/working/eeg_model_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X = np.load(DATA_DIR / "X.npy")          # [N, T, C]
y = np.load(DATA_DIR / "y.npy")          # [N]
subjects = np.load(DATA_DIR / "subjects.npy")
meta_df = pd.read_csv(DATA_DIR / "meta.csv")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("subjects shape:", subjects.shape)
print("Unique subjects:", np.unique(subjects))
print("Class counts:", pd.Series(y).value_counts().sort_index().to_dict())

# -------------------------------
# Subject-wise split
# -------------------------------
# Fixed split for first experiment:
# Train = A01-A06, Val = A07-A08, Test = A09
train_subjects = ["A01", "A02", "A03", "A04", "A05", "A06"]
val_subjects   = ["A07", "A08"]
test_subjects  = ["A09"]

train_mask = np.isin(subjects, train_subjects)
val_mask   = np.isin(subjects, val_subjects)
test_mask  = np.isin(subjects, test_subjects)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val     = X[val_mask], y[val_mask]
X_test, y_test   = X[test_mask], y[test_mask]

print("\nSplit sizes:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# -------------------------------
# Dataset class
# -------------------------------
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)   # [N, T, C]
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_ds = EEGDataset(X_train, y_train)
val_ds   = EEGDataset(X_val, y_val)
test_ds  = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN_MEMORY)

# -------------------------------
# Plot style helper
# -------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

CLASS_NAMES = ["left_hand", "right_hand", "feet", "tongue"]

Device: cuda
X shape: (2592, 1000, 22)
y shape: (2592,)
subjects shape: (2592,)
Unique subjects: ['A01' 'A02' 'A03' 'A04' 'A05' 'A06' 'A07' 'A08' 'A09']
Class counts: {0: 648, 1: 648, 2: 648, 3: 648}

Split sizes:
Train: (1728, 1000, 22) (1728,)
Val  : (576, 1000, 22) (576,)
Test : (288, 1000, 22) (288,)


In [2]:
# ============================================
# EEG Modelling — Cell 2
# CNN-LSTM + Transformer + train/eval
# ============================================

# -------------------------------
# Model 1: CNN-LSTM
# -------------------------------
class SafeLSTM(nn.LSTM):
    def flatten_parameters(self):
        return

class CNNLSTM(nn.Module):
    def __init__(self, n_channels=22, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        self.lstm = SafeLSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.classifier(x)
        return x

# -------------------------------
# Model 2: Transformer
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class EEGTransformer(nn.Module):
    def __init__(self, n_channels=22, n_classes=4, d_model=64, nhead=4, num_layers=3, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_channels, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=1200)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        # x: [B, T, C]
        x = self.input_proj(x)      # [B, T, d_model]
        x = self.pos_enc(x)
        x = self.encoder(x)
        x = self.norm(x)
        x = x.mean(dim=1)           # global average over time
        x = self.head(x)
        return x

# -------------------------------
# Train/eval helpers
# -------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    losses = []
    all_preds, all_targets = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)

        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(yb.detach().cpu().numpy())

    loss_mean = float(np.mean(losses))
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average="macro")
    kappa = cohen_kappa_score(all_targets, all_preds)

    return {
        "loss": loss_mean,
        "acc": acc,
        "f1": f1,
        "kappa": kappa,
        "y_true": np.array(all_targets),
        "y_pred": np.array(all_preds)
    }

def train_model(model, train_loader, val_loader, name, epochs=20, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    history = []
    best_f1 = -1
    best_path = OUT_DIR / f"best_{name}.pt"

    for epoch in range(1, epochs + 1):
        tr = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        va = run_epoch(model, val_loader, criterion, optimizer=None)

        history.append({
            "epoch": epoch,
            "train_loss": tr["loss"],
            "train_acc": tr["acc"],
            "train_f1": tr["f1"],
            "train_kappa": tr["kappa"],
            "val_loss": va["loss"],
            "val_acc": va["acc"],
            "val_f1": va["f1"],
            "val_kappa": va["kappa"],
        })

        print(
            f"[{name}] Epoch {epoch:02d} | "
            f"train_loss={tr['loss']:.4f} train_acc={tr['acc']:.4f} train_f1={tr['f1']:.4f} | "
            f"val_loss={va['loss']:.4f} val_acc={va['acc']:.4f} val_f1={va['f1']:.4f}"
        )

        if va["f1"] > best_f1:
            best_f1 = va["f1"]
            torch.save(model.state_dict(), best_path)

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUT_DIR / f"history_{name}.csv", index=False)

    model.load_state_dict(torch.load(best_path, map_location=device))
    test_metrics = run_epoch(model, test_loader, criterion, optimizer=None)

    return hist_df, test_metrics, best_path

def save_training_curves(hist_df, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train", color="#4C78A8", linewidth=2)
    axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="Val", color="#E45756", linewidth=2)
    axes[0].set_title(f"{model_name}: Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    paper_axes(axes[0])

    axes[1].plot(hist_df["epoch"], hist_df["train_f1"], label="Train", color="#54A24B", linewidth=2)
    axes[1].plot(hist_df["epoch"], hist_df["val_f1"], label="Val", color="#F58518", linewidth=2)
    axes[1].set_title(f"{model_name}: Macro-F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro-F1")
    axes[1].legend()
    paper_axes(axes[1])

    fig.tight_layout()
    out = OUT_DIR / f"curve_{model_name}.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

def save_confusion(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{model_name}: Test Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    out = OUT_DIR / f"cm_{model_name}.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

# -------------------------------
# Train both models
# -------------------------------
cnn_lstm = CNNLSTM(n_channels=22, n_classes=4).to(device)
transformer = EEGTransformer(n_channels=22, n_classes=4).to(device)

hist_cnn, test_cnn, path_cnn = train_model(cnn_lstm, train_loader, val_loader, name="cnn_lstm", epochs=20, lr=1e-3)
hist_trf, test_trf, path_trf = train_model(transformer, train_loader, val_loader, name="transformer", epochs=20, lr=1e-3)

# -------------------------------
# Save curves + confusion matrices
# -------------------------------
save_training_curves(hist_cnn, "cnn_lstm")
save_training_curves(hist_trf, "transformer")

save_confusion(test_cnn["y_true"], test_cnn["y_pred"], "cnn_lstm")
save_confusion(test_trf["y_true"], test_trf["y_pred"], "transformer")

# -------------------------------
# Final result table
# -------------------------------
results_df = pd.DataFrame([
    {
        "model": "CNN-LSTM",
        "test_acc": test_cnn["acc"],
        "test_macro_f1": test_cnn["f1"],
        "test_kappa": test_cnn["kappa"],
        "best_ckpt": str(path_cnn)
    },
    {
        "model": "Transformer",
        "test_acc": test_trf["acc"],
        "test_macro_f1": test_trf["f1"],
        "test_kappa": test_trf["kappa"],
        "best_ckpt": str(path_trf)
    }
])

results_csv = OUT_DIR / "model_results.csv"
results_df.to_csv(results_csv, index=False)

print("\nFinal test results:")
display(results_df)

print("\nCNN-LSTM classification report:")
print(classification_report(test_cnn["y_true"], test_cnn["y_pred"], target_names=CLASS_NAMES, digits=4))

print("\nTransformer classification report:")
print(classification_report(test_trf["y_true"], test_trf["y_pred"], target_names=CLASS_NAMES, digits=4))

print("Saved:", results_csv)

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
